# Scraping database review

This notebook browses `scraping.db`, the scraping module's only implemented database. The top-level `src/storage/` module is still a skeleton, so it has no schema or data to review.

> **Kernel note:** select the project's `.venv` Python interpreter/kernel in VS Code so `pandas` and `src.scraping` imports resolve.

In [19]:
import json
import sqlite3
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    """Find the repository regardless of the notebook launch directory."""
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'config.yaml').exists():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DB_PATH = REPO_ROOT / 'scraping.db'
print(f'Repository: {REPO_ROOT}')
print(f'Database:   {DB_PATH} ({DB_PATH.stat().st_size if DB_PATH.exists() else 0:,} bytes)')

Repository: /Users/kumo/programming/competitor_product_search
Database:   /Users/kumo/programming/competitor_product_search/scraping.db (2,338,816 bytes)


In [20]:
from src.scraping.storage import ScrapeDB

db = ScrapeDB(DB_PATH)
db.init_db()  # Idempotent: creates the six tables on a fresh, empty database.

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name",
    db.conn,
)
table_summary = pd.DataFrame(
    [
        {'table': name, 'rows': db.conn.execute(f'SELECT COUNT(*) FROM {name}').fetchone()[0]}
        for name in tables['name']
    ]
)
display(table_summary)

,table,rows
0,escalations,0
1,golden_samples,5
2,invalid_target_phrases,0
3,parsers,1
4,results,0
5,scrape_runs,0


## `scrape_runs`

The operational log: URL, scraper path, outcome, latency, cost, and the parser used when applicable.

In [21]:
runs = pd.read_sql_query('SELECT * FROM scrape_runs ORDER BY id DESC', db.conn)
display(runs)

runs_with_parser = pd.read_sql_query(
    """
    SELECT r.*, p.site AS parser_site, p.version AS parser_version
    FROM scrape_runs AS r
    LEFT JOIN parsers AS p ON p.id = r.winning_parser_id
    ORDER BY r.id DESC
    """,
    db.conn,
)
display(runs_with_parser)

,id,url,host,site,scraper,scraped_at,outcome,path,winning_parser_id,attempts,model_used,latency_ms,cost


,id,url,host,site,scraper,scraped_at,outcome,path,winning_parser_id,attempts,model_used,latency_ms,cost,parser_site,parser_version


## `results`

This is the main scraped-product data table. The overview keeps JSON compact; the second view expands `product_data` into its individual ProductData fields.

In [22]:
def shorten(value: object, limit: int = 180) -> object:
    if value is None or pd.isna(value):
        return value
    text = str(value)
    return text if len(text) <= limit else text[:limit] + ' …'


def decode_json(value: object) -> dict:
    if not value:
        return {}
    try:
        decoded = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return {'_unparseable_product_data': value}
    return decoded if isinstance(decoded, dict) else {'_json_value': decoded}


results = pd.read_sql_query('SELECT * FROM results ORDER BY id DESC', db.conn)
results_overview = results.copy()
if 'product_data' in results_overview:
    results_overview['product_data'] = results_overview['product_data'].map(shorten)
display(results_overview)

result_metadata = results.reindex(columns=['id', 'url', 'site', 'scraped_at'])
result_fields = pd.json_normalize(results['product_data'].map(decode_json).tolist())
results_flat = result_metadata.join(result_fields)
display(results_flat)

,id,url,site,scraped_at,product_data


,id,url,site,scraped_at


## `parsers`

Generated parser code is intentionally shortened here. Use `show_blob` below to inspect a full parser.

In [23]:
parsers = pd.read_sql_query('SELECT * FROM parsers ORDER BY id DESC', db.conn)
parsers_overview = parsers.copy()
if 'code' in parsers_overview:
    parsers_overview['code'] = parsers_overview['code'].map(shorten)
display(parsers_overview)

,id,site,version,code,page_type_scope,status,created_at,created_by
0,1,tesco,cs_20260803_211813,"def parse(html: str, url: str) -> dict:\n import json\n import re\n from bs4 import BeautifulSoup\n\n soup = BeautifulSoup(html, 'lxml')\n re...",None,active,2026-08-03T21:18:13.588Z,initial


## `golden_samples`

Golden snapshots validate newly repaired parsers. The flattened view makes expected ProductData fields easy to compare.

In [24]:
goldens = pd.read_sql_query('SELECT * FROM golden_samples ORDER BY id DESC', db.conn)
goldens_overview = goldens.copy()
for column in ['html_snapshot', 'expected_output']:
    if column in goldens_overview:
        goldens_overview[column] = goldens_overview[column].map(shorten)
display(goldens_overview)

golden_metadata = goldens.reindex(columns=['id', 'site', 'page_type', 'captured_at', 'is_stale'])
golden_fields = pd.json_normalize(goldens['expected_output'].map(decode_json).tolist())
goldens_flat = golden_metadata.join(golden_fields)
display(goldens_flat)

,id,site,page_type,html_snapshot,expected_output,captured_at,is_stale,created_by
0,5,tesco,discounted,"<!DOCTYPE html>\n<!-- VERSION: {""mfe-orchestrator"":""2026.07.29-b0b5663d"",""mfe-basket-manager"":""2026.07.29-187409bf"",""mfe-analytics"":""2026.07.30-d5061e36"",""m...","{""url"": ""https://www.tesco.com/shop/en-GB/products/330671337"", ""website"": ""tesco"", ""scraped_at"": ""2026-08-03T21:12:27.910741Z"", ""source_type"": ""html"", ""pars...",2026-08-03T21:18:13.631Z,0,coldstart
1,4,tesco,discounted,"<!DOCTYPE html>\n<!-- VERSION: {""mfe-orchestrator"":""2026.07.29-b0b5663d"",""mfe-basket-manager"":""2026.07.29-187409bf"",""mfe-global-scripts"":""2026.06.23-2535bc8...","{""url"": ""https://www.tesco.com/shop/en-GB/products/330702828"", ""website"": ""tesco"", ""scraped_at"": ""2026-08-03T21:12:27.663348Z"", ""source_type"": ""html"", ""pars...",2026-08-03T21:18:13.620Z,0,coldstart
2,3,tesco,out_of_stock,"<!DOCTYPE html>\n<!-- VERSION: {""mfe-orchestrator"":""2026.07.29-b0b5663d"",""mfe-basket-manager"":""2026.07.29-187409bf"",""mfe-analytics"":""2026.07.30-d5061e36"",""m...","{""url"": ""https://www.tesco.com/shop/en-GB/products/305728358"", ""website"": ""tesco"", ""scraped_at"": ""2026-08-03T21:12:27.414603Z"", ""source_type"": ""html"", ""pars...",2026-08-03T21:18:13.609Z,0,coldstart
3,2,tesco,standard,"<!DOCTYPE html>\n<!-- VERSION: {""mfe-orchestrator"":""2026.07.29-b0b5663d"",""mfe-basket-manager"":""2026.07.29-187409bf"",""mfe-analytics"":""2026.07.30-d5061e36"",""m...","{""url"": ""https://www.tesco.com/shop/en-GB/products/322999875"", ""website"": ""tesco"", ""scraped_at"": ""2026-08-03T21:12:27.134682Z"", ""source_type"": ""html"", ""pars...",2026-08-03T21:18:13.597Z,0,coldstart
4,1,tesco,standard,"<!DOCTYPE html>\n<!-- VERSION: {""mfe-orchestrator"":""2026.07.29-b0b5663d"",""mfe-analytics"":""2026.07.30-d5061e36"",""mfe-global-scripts"":""2026.06.23-2535bc8e"",""m...","{""url"": ""https://www.tesco.com/shop/en-GB/products/325502056"", ""website"": ""tesco"", ""scraped_at"": ""2026-08-03T21:12:26.757954Z"", ""source_type"": ""html"", ""pars...",2026-08-03T21:18:13.592Z,0,coldstart


,id,site,page_type,captured_at,is_stale,url,website,scraped_at,source_type,parser_version,title,brand,gtin,image_urls,variant,price,currency,list_price,membership_price,unit_price,unit,in_stock,availability_raw,raw
0,5,tesco,discounted,2026-08-03T21:18:13.631Z,0,https://www.tesco.com/shop/en-GB/products/330671337,tesco,2026-08-03T21:12:27.910741Z,html,coldstart_v1,PureMate 43-Inch Oscillating Bladeless Tower Fan with Remote Control (Black),Puremate,05060744147602,"[https://digitalcontent.api.tesco.com/v2/media/marketplace/2c5e3d25-8f3b-4910-8376-24dbc8778bbb/38dcff6e4bee4658a2a3f193a86c1e13_1537823179.jpeg, https://di...",None,119.99,GBP,169.99,None,None,None,True,In stock,None
1,4,tesco,discounted,2026-08-03T21:18:13.620Z,0,https://www.tesco.com/shop/en-GB/products/330702828,tesco,2026-08-03T21:12:27.663348Z,html,coldstart_v1,EMtronics 2500W Double Plate Stove - Black,EMtronics,05056149881647,"[https://digitalcontent.api.tesco.com/v2/media/marketplace/181b23ed-3c07-4962-bd0a-eb21d286a6c0/2da8c29148b347058d94c7af0053d986_1168367900.jpeg, https://di...",None,29.99,GBP,39.99,None,None,None,True,In stock,None
2,3,tesco,out_of_stock,2026-08-03T21:18:13.609Z,0,https://www.tesco.com/shop/en-GB/products/305728358,tesco,2026-08-03T21:12:27.414603Z,html,coldstart_v1,Youngs Gastro 2 Tempura Battered Fish Fillets 270G,YOUNGS,05000205048284,[https://digitalcontent.api.tesco.com/v2/media/ghs/f8326a76-6678-4705-bf3f-70141b200ae3/2ed5a0f6-b6dc-4dec-9013-82b2848223ad.jpeg?h=225&w=225],None,5.25,GBP,NaN,None,None,None,False,Out of stock,None
3,2,tesco,standard,2026-08-03T21:18:13.597Z,0,https://www.tesco.com/shop/en-GB/products/322999875,tesco,2026-08-03T21:12:27.134682Z,html,coldstart_v1,L’Oreal Elvive Colour Protect Shampoo 700ml,L'ORÉAL,03600524140144,"[https://digitalcontent.api.tesco.com/v2/media/ghs/71872d0e-4713-47d2-ba8a-74b46b3209bd/38835d87-8337-49d6-8f67-abe5be53cd12_1380479791.jpeg, https://digita...",None,5.75,GBP,NaN,None,None,None,True,In stock,None
4,1,tesco,standard,2026-08-03T21:18:13.592Z,0,https://www.tesco.com/shop/en-GB/products/325502056,tesco,2026-08-03T21:12:26.757954Z,html,coldstart_v1,GEEPAS 1.7L Illuminating Electric Glass Kettle 2200W,Geepas,06294015555776,"[https://digitalcontent.api.tesco.com/v2/media/marketplace/547b2bc7-e07e-4183-be13-ce681878b90b/8ed44977de104766b218e633c4916b25_1976097602.jpeg, https://di...",None,18.99,GBP,NaN,None,None,None,True,In stock,None


## `escalations`

Escalations are deduplicated by signature. The second view uses the app's store API for the currently open queue.

In [25]:
from src.scraping.storage import EscalationStore

escalations = pd.read_sql_query('SELECT * FROM escalations ORDER BY id DESC', db.conn)
escalations_overview = escalations.copy()
if 'snapshot' in escalations_overview:
    escalations_overview['snapshot'] = escalations_overview['snapshot'].map(shorten)
display(escalations_overview)

open_escalations = pd.DataFrame(EscalationStore(db).get_open())
display(open_escalations)

,id,signature,reason,affected_count,snapshot,status,created_at


""


## `invalid_target_phrases`

Small lookup table used to recognize pages that are not product targets.

In [26]:
phrases = pd.read_sql_query('SELECT * FROM invalid_target_phrases ORDER BY id DESC', db.conn)
display(phrases)

,id,site,phrase,source,added_at


## Full-field drill-down

Overview tables truncate long code and snapshots. Call `show_blob` with a table, row id, and column to print the complete value; JSON is formatted for readability.

In [27]:
REVIEW_TABLES = {
    'parsers', 'golden_samples', 'scrape_runs', 'results', 'escalations', 'invalid_target_phrases'
}


def show_blob(table: str, row_id: int, column: str) -> None:
    """Print one complete text/JSON field from a reviewed table."""
    if table not in REVIEW_TABLES:
        raise ValueError(f'Unknown review table: {table}')
    valid_columns = {row['name'] for row in db.conn.execute(f'PRAGMA table_info({table})')}
    if column not in valid_columns:
        raise ValueError(f'Unknown column for {table}: {column}')

    row = db.conn.execute(
        f'SELECT {column} FROM {table} WHERE id = ?', (row_id,)
    ).fetchone()
    if row is None:
        raise LookupError(f'No {table} row with id={row_id}')

    value = row[0]
    if value is None:
        print('(NULL)')
        return
    try:
        print(json.dumps(json.loads(value), indent=2, ensure_ascii=False, default=str))
    except (TypeError, json.JSONDecodeError):
        print(value)


# Examples:
# show_blob('parsers', 1, 'code')
# show_blob('golden_samples', 1, 'html_snapshot')
# show_blob('results', 1, 'product_data')
# show_blob('escalations', 1, 'snapshot')

## Handy filtered questions

For common site-level questions, the purpose-built stores are more convenient than writing the aggregation again. Change `SITE` and re-run this cell.

In [ ]:
from src.scraping.storage import ParserStore, RunStore

SITE = 'tesco'
active_parsers = pd.DataFrame(ParserStore(db).get_active_ordered_by_hits(SITE))
parser_hit_rates = pd.DataFrame(RunStore(db).get_hit_rates(SITE))

print(f'Active parsers for {SITE}:')
display(active_parsers)
print(f'Parser hit rates for {SITE}:')
display(parser_hit_rates)